# Image Cell Counting

In [ ]:
import torch
import gc
assert torch.cuda.is_available(), "GPU not available" #this will give an error if no GPU available 

In [ ]:
!pip install roboflow
!pip install ultralytics

In [ ]:
from google.colab import userdata
from roboflow import Roboflow
import os
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import numpy as np
import cv2
from getpass import getpass
from ultralytics import YOLO
from IPython.display import Image

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
from pathlib import Path
OUT = Path("/content/drive/MyDrive/aug_study")
OUT.mkdir(parents=True, exist_ok=True)
LOG = OUT / "aug_study_log.csv"

In [ ]:
# api_key = userdata.get("ROBOFLOW_API_KEY") #for Google colab UI
api_key = getpass("Roboflow API key: ") #for Google colab implementation in VS code 

In [ ]:
rf = Roboflow(api_key=api_key) # *---- Add api key to secrect w name ROBOFLOW_API_KEY ----*
project = rf.workspace("pranjal-singh-evqc5").project(
    "cell-counting-zeqo5-raras"  # *-------- Need to change this to specific dir --------*
)

version = project.version(1) #version generated on roboflow universe
dataset = version.download("yolo26") #chose this format for PyTorch compatibality

In [ ]:
print(os.listdir(dataset.location))
assert "data.yaml" in os.listdir(dataset.location), "Incorrect data format, download as yolo"

## No augmentation Baseline

In [ ]:
model = YOLO("yolo26n.pt")

In [ ]:
DATA = f"{dataset.location}/data.yaml" #data location for training 

In [ ]:
results_log = [] #to compare augmentation results 

"""
Check Ultralytics docs for more details on each parameter: https://docs.ultralytics.com/guides/yolo-data-augmentation
"""
off = dict(
    hsv_h = 0, #hue adjustment
    hsv_s = 0, #saturation adjustment
    hsv_v = 0, #brightness adjustment 
    degrees = 0, #rotation
    translate = 0, #traslation
    scale = 0, #image resize
    shear= 0, #image skewing
    perspective=0, #prespective transformation along x & y axis
    flipud= 0, #flip up and down
    fliplr = 0, #flip right and left
    bgr = 0, #swaps im,age from RGB to BGR
    mosaic = 0, #conmbines four train image into one
    mixup = 0, #blends two images
    cutmix = 0, #cuts rectangle from one image and pastes onto anothe
    copy_paste = 0, #copies objects within or between images #determines method used for copy-pate augmentation     
)

fixed = dict( #fixing the lr, batch size, and optimizer so it doen't span all options on auto
    data=DATA, imgsz=640, single_cls=True,
    optimizer="AdamW", lr0=0.001,   # <- replace with what Step 0 showed
    batch=16, workers=8, plots=False,
)

fast = dict(cache="ram", val=False)   # if RAM is tight

screen = dict(epochs=25, fraction=0.5)   # cheap screening schedule
full   = dict(epochs=50, fraction=1.0)   # final schedule


### Baseline training


In [ ]:
name = "scr_baseline"
seed = 0
overrides = {} #overrides whatever parameter is in here with the "OFF" value

model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)   # now saved to Google Drive
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## ONLY Color Augmentation

In [ ]:
name = "scr_hsv"
seed = 0
overrides = dict(hsv_h=0.015, hsv_s=0.7, hsv_v=0.4) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Flip Left Right

In [ ]:
name = "scr_fliplr"
seed = 0
overrides = dict(fliplr=0.5) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Flip up and down

In [ ]:
name = "scr_flipyd"
seed = 0
overrides = dict(flipud=0.5) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Transtlate

In [ ]:
name = "scr_trans"
seed = 0
overrides = dict(translate=0.1) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Scale

In [ ]:
name = "scr_scale"
seed = 0
overrides = dict(scale=0.5) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Rotate

In [ ]:
name = "scr_rotate"
seed = 0
overrides = dict(degrees=90) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Shear and Perspective

In [ ]:
name = "scr_skew"
seed = 0
overrides = dict(shear=5, perspective=0.0005) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Mosaic 

In [ ]:
name = "scr_mosac"
seed = 0
overrides = dict(mosaic=1.0, close_mosaic=10) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Mixup 

In [ ]:
name = "scr_mixup"
seed = 0
overrides = dict(mixup=0.2) #overrides whatever parameter is in here with the "OFF" value

model = YOLO("yolo26n.pt")
model.train(**fixed, **fast, **screen, seed=seed,
            project=f"{OUT}", name=f"{name}_s{seed}", exist_ok=True,
            **{**off, **overrides})

In [ ]:
m = YOLO(OUT / f"{name}_s{seed}" / "weights" / "last.pt").val(data=DATA, split="val", plots=False)

results_log.append(dict(stage="screen", name=name, seed=seed,
                        map50=m.box.map50, map5095=m.box.map,
                        precision=m.box.mp, recall=m.box.mr))

pd.DataFrame(results_log).to_csv(LOG, index=False)
del model; gc.collect(); torch.cuda.empty_cache()
pd.DataFrame(results_log)

## Plotting augmentation exploration results 

In [ ]:
LOG = "/content/drive/MyDrive/aug_study/aug_study_log.csv"
df = pd.read_csv(LOG)
df

In [ ]:
precision = df["precision"]
recall = df["recall"]

f1_score = 2*(precision * recall)/(precision + recall)

print(f1_score)

In [ ]:
agumen_labels = df["name"]

fig,ax = plt.subplots(figsize=(12,4))
ax.bar(agumen_labels,f1_score, color= "navy")
ax.axhline(y=f1_score[0],linestyle="--")
ax.set_ylim(0.85,1.0)
ax.set_xlabel("Augmentation")
ax.set_ylabel("F1 score")
plt.show()